In [ ]:
import json
from pathlib import Path
import hashlib

import pandas as pd
import polars as pl
import yaml
import numpy as np
import structlog

from common import read_day, daterange

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

log = structlog.get_logger()

In [3]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}/id_lookup"

create test dataframe

In [5]:
test = pl.DataFrame(
    {
        "device_aid_hashed": [
            hashlib.md5("foo".encode("utf-8")).hexdigest(),
            hashlib.md5("bar".encode("utf-8")).hexdigest(),
            hashlib.md5("bar".encode("utf-8")).hexdigest(),
            hashlib.md5("bar".encode("utf-8")).hexdigest(),
            hashlib.md5("bar".encode("utf-8")).hexdigest(),
        ],
        "device_aid_type": ["AAID", "AAID", "AAID", "IDFA", "IDFA"],
    }
)
test

device_aid_hashed,device_aid_type
str,str
"""acbd18db4cc2f85cedef654fccc4a4…","""AAID"""
"""37b51d194a7513e45b56f6524f2d51…","""AAID"""
"""37b51d194a7513e45b56f6524f2d51…","""AAID"""
"""37b51d194a7513e45b56f6524f2d51…","""IDFA"""
"""37b51d194a7513e45b56f6524f2d51…","""IDFA"""


In [22]:
test_lookup = test.unique(
    subset=["device_aid_hashed", "device_aid_type"], keep="first", maintain_order=True
).with_row_index("device_id", offset=1)
test_lookup

device_id,device_aid_hashed,device_aid_type,id_with_type
u32,str,str,str
1,"""acbd18db4cc2f85cedef654fccc4a4…","""AAID""","""acbd18db4cc2f85cedef654fccc4a4…"
2,"""37b51d194a7513e45b56f6524f2d51…","""AAID""","""37b51d194a7513e45b56f6524f2d51…"
3,"""37b51d194a7513e45b56f6524f2d51…","""IDFA""","""37b51d194a7513e45b56f6524f2d51…"


In [ ]:
mapping = dict(
    zip(
        test_lookup.select(["device_aid_hashed", "device_aid_type"]).iter_rows(),
        test_lookup["device_id"].to_list(),
    )
)

In [9]:
mapping

{('acbd18db4cc2f85cedef654fccc4a4d8', 'AAID'): 1,
 ('37b51d194a7513e45b56f6524f2d51f2', 'AAID'): 2,
 ('37b51d194a7513e45b56f6524f2d51f2', 'IDFA'): 3}

Good, but cannot be serialized as JSON, because a tuple cannot be key.

In [ ]:
test = test.with_columns(
    pl.concat_str(
        [pl.col("device_aid_hashed"), pl.col("device_aid_type")], separator="."
    ).alias("id_with_type")
)

In [ ]:
test_lookup = (
    test.unique(subset=["id_with_type"], keep="first")
    .sort("id_with_type")
    .with_row_index("device_id", offset=1)
)
test_lookup

device_id,device_aid_hashed,device_aid_type,id_with_type
u32,str,str,str
1,"""37b51d194a7513e45b56f6524f2d51…","""AAID""","""37b51d194a7513e45b56f6524f2d51…"
2,"""37b51d194a7513e45b56f6524f2d51…","""IDFA""","""37b51d194a7513e45b56f6524f2d51…"
3,"""acbd18db4cc2f85cedef654fccc4a4…","""AAID""","""acbd18db4cc2f85cedef654fccc4a4…"


In [20]:
mapping = dict(zip(test_lookup["id_with_type"], test_lookup["device_id"]))

In [21]:
mapping

{'37b51d194a7513e45b56f6524f2d51f2.AAID': 1,
 '37b51d194a7513e45b56f6524f2d51f2.IDFA': 2,
 'acbd18db4cc2f85cedef654fccc4a4d8.AAID': 3}

In [3]:
ids = []
for i in days[:3]:
    temp = read_day(i, config["dirs"]["raw_source"])
    dids = temp.unique(subset=["device_aid_hashed", "device_aid_type"], keep="first")[
        "device_aid_hashed"
    ].to_list()
    ids += dids

In [4]:
temp.unique(subset=["device_aid_hashed"], keep="first").select(
    ["device_aid_hashed", "device_aid_type"]
)

device_aid_hashed,device_aid_type
str,str
"""88157fc7f7595dc27863c8b1d2c2f8…","""IDFA"""
"""afa17b5dadbc961037686a34fdaae0…","""IDFA"""
"""228352c7852b5b84a6c3d6100cfd79…","""AAID"""
"""fd840ad60e744d2fe42287a9e6ae73…","""AAID"""
"""a5fed4865519f6fddfbd251f4e5fdf…","""AAID"""
…,…
"""f322f8f2db7ce9f0cd62e505fdc93b…","""AAID"""
"""fc6ff7a8bddf54dbd631a6f3cc49b4…","""IDFA"""
"""e484f531f77b218c963fd3eff73053…","""AAID"""


In [ ]:
temp.unique(subset=["device_aid_hashed", "device_aid_type"], keep="first").select(
    ["device_aid_hashed", "device_aid_type"]
)

device_aid_hashed,device_aid_type
str,str
"""a70830cb78731b00f81eb293e90362…","""AAID"""
"""3462330928ee6ec8093fc06355234c…","""AAID"""
"""6b41de7362c19ce8380a29a60122f6…","""AAID"""
"""b67b640af32ace46f413c9771a3a3b…","""IDFA"""
"""284df8fcdb9d73c294b29da3b68607…","""AAID"""
…,…
"""6c715fbc0384ec3e3b5328a1930078…","""AAID"""
"""9a0df0612e08f02136c896bd99d298…","""IDFA"""
"""a7263d33e181cb27bc8a481ba7c802…","""IDFA"""


In [ ]:
def collect_ids(raw_pings_path: str, id_column: str, id_type_column: str) -> list[str]:
    ids = []
    for i in days:
        temp = read_day(i, raw_pings_path)
        dids = temp.unique(subset=[id_column, id_type_column], keep="first")[
            id_column
        ].to_list()
        ids += dids
    return ids


def create_lookup(ids: list[str]) -> dict:
    lookup = {}
    for i, device_id in enumerate(np.unique(ids)):
        lookup[device_id] = i + 1
    return lookup


def create_partitions(lookup: dict[str, int]):
    return {v: k[:2] for k, v in lookup.items()}

In [4]:
try:
    with open(f"{OUTPUT}/id_lookup.json", "r") as fp:
        lookup = json.load(fp)
except FileNotFoundError:
    # runs 30 minutes for Budapest data
    ids = collect_ids(config["dirs"]["raw_source"], id_column=config["id_column"])
    log.info(f"number unique devices: {len(np.unique(ids))}")

    lookup = create_lookup(ids)

    Path(OUTPUT).mkdir(parents=True, exist_ok=True)
    with open(f"{OUTPUT}/id_lookup.json", "w") as fp:
        json.dump(lookup, fp, indent=1)

2026-04-30 14:00:54 [info     ] number unique devices: 1652460


## create partitions from original IDs

In [5]:
partition = create_partitions(lookup)

with open(f"{OUTPUT}/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

### descriptives

In [11]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)
partition_distribution["device_id"].describe()

count    256.000000
mean     377.832031
std       19.991349
min      309.000000
25%      364.750000
50%      376.500000
75%      392.000000
max      424.000000
Name: device_id, dtype: float64

In [ ]:
with open(
    "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest_old/id_lookup/id_lookup.json",
    "r",
) as fp:
    lu_old = json.load(fp)

In [27]:
len(lu_old)

1652460